## <V&A(빅토리아 앤 알버트 박물관) 데이터 수집>
- API 설명: https://developers.vam.ac.uk/

### 1. 필요한 것들 불러오기

In [1]:
import os
import time
import requests
import pandas as pd

MUSEUM_CODE = "VA"
OUTPUT_EXCEL = f"../data/{MUSEUM_CODE}_hyungbae.xlsx"
HEADERS = {"User-Agent": "aks-digital-humanities-research/1.0"}

### 2. API 호출해서 원하는 정보 얻기

In [2]:
CATEGORY = "흉배"
CATEGORY_LETTER = "H"
KEYWORD = "rank badge"

rows = []
serial_counter = {}
failed_ids = []

def next_temp_id(museum_code):
    n = serial_counter.get(museum_code, 0) + 1
    serial_counter[museum_code] = n
    return f"Y{museum_code}{CATEGORY_LETTER}{n:02d}"

def search_va(keyword=KEYWORD, page_size=45):
    base = "https://api.vam.ac.uk/v2/objects/search"
    page = 1
    system_numbers = []
    while True:
        params = {"q": keyword, "page_size": page_size, "page": page}
        resp = requests.get(base, params=params, headers=HEADERS, timeout=20)
        resp.raise_for_status()
        payload = resp.json()
        records = payload.get("records") or []
        if not records:
            break
        for rec in records:
            object_type = rec.get("objectType") or ""
            if keyword.lower() in object_type.lower():   # 검색 단계에서 바로 필터링
                system_numbers.append(rec["systemNumber"])

        total_pages = (payload.get("info") or {}).get("pages") or page
        if page >= total_pages:
            break
        page += 1
        time.sleep(0.3)
    return system_numbers

def fetch_va_detail(system_number):
    url = f"https://api.vam.ac.uk/v2/museumobject/{system_number}"
    resp = requests.get(url, headers=HEADERS, timeout=20)
    resp.raise_for_status()
    return resp.json().get("record") or {}

def fetch_va(keyword=KEYWORD):
    system_numbers = search_va(keyword)
    print(f"objectType 기준 필터 후 대상: {len(system_numbers)}건")

    for sn in system_numbers:
        time.sleep(0.3)
        try:
            item = fetch_va_detail(sn)
        except (requests.exceptions.RequestException, ValueError) as e:
            print(f"{sn} 실패: {e}")
            failed_ids.append(sn)
            continue

        temp_id = next_temp_id(MUSEUM_CODE)

        image_ids = item.get("images") or []
        image_urls = [f"https://framemark.vam.ac.uk/collections/{img_id}/full/full/0/default.jpg" for img_id in image_ids]

        materials = "; ".join(m.get("text", "") for m in (item.get("materials") or []) if m.get("text"))
        techniques = "; ".join(t.get("text", "") for t in (item.get("techniques") or []) if t.get("text"))
        dims = item.get("dimensions") or []
        dimensions_str = "; ".join(
            f"{d.get('dimension','')}: {d.get('value','')}{d.get('unit','')}".strip()
            for d in dims if d.get("value")
        )
        places = item.get("placesOfOrigin") or []
        place_str = "; ".join(p.get("place", {}).get("text", "") for p in places if p.get("place"))
        dates = item.get("productionDates") or []
        date_str = "; ".join(d.get("date", {}).get("text", "") for d in dates if d.get("date"))

        rows.append({
            "임시ID": temp_id,
            "분류": CATEGORY,
            "소장처": "빅토리아 앤 알버트 박물관",
            "소장처유물번호": item.get("accessionNumber"),
            "한글명": "",
            "한자명": "",
            "영어명": item.get("_primaryTitle") or item.get("objectType"),
            "URL": f"https://collections.vam.ac.uk/item/{sn}/",
            "searched_keyword": keyword,
            "objectType": item.get("objectType"),
            "place": place_str,
            "date": date_str,
            "materials": materials,
            "techniques": techniques,
            "dimensions": dimensions_str,
            "briefDescription": item.get("briefDescription"),
            "summaryDescription": item.get("summaryDescription"),
            "physicalDescription": item.get("physicalDescription"),
            "objectHistory": item.get("objectHistory"),
            "creditLine": item.get("creditLine"),
            "image_url": image_urls[0] if image_urls else None,
            "image_urls": "; ".join(image_urls),
            "license_note": "V&A Collections (건별 라이선스 확인 필요)",
        })

    if failed_ids:
        print(f"총 {len(failed_ids)}건 실패: {failed_ids}")

fetch_va()
print(f"{len(rows)}건 수집 완료")

objectType 기준 필터 후 대상: 50건
50건 수집 완료


### 3. 데이터프레임 -> 엑셀

In [3]:
df = pd.DataFrame(rows)
master_cols = ["임시ID", "분류", "소장처", "소장처유물번호", "한글명", "한자명", "영어명",
               "URL", "searched_keyword", "objectType", "place", "date", "materials", "techniques",
               "dimensions", "briefDescription", "summaryDescription", "physicalDescription",
               "objectHistory", "creditLine", "image_url", "image_urls", "license_note"]
other_cols = [c for c in df.columns if c not in master_cols]
df = df[master_cols + other_cols]

pure_hyungbae = df[df["영어명"].str.contains("rank badge", case=False, na=False)]
pure_hyungbae.to_excel(OUTPUT_EXCEL, index=False)

os.makedirs("../data", exist_ok=True)
pure_hyungbae.to_excel(OUTPUT_EXCEL, index=False)

print(pure_hyungbae.shape)
pure_hyungbae.head()

(50, 23)


,임시ID,분류,소장처,소장처유물번호,한글명,한자명,영어명,URL,searched_keyword,objectType,...,techniques,dimensions,briefDescription,summaryDescription,physicalDescription,objectHistory,creditLine,image_url,image_urls,license_note
0,YVAH01,흉배,빅토리아 앤 알버트 박물관,FE.11-1986,,,Rank badge,https://collections.vam.ac.uk/item/O14485/,rank badge,Rank badge,...,tapestry,Height: 32cm; Width: 35cm,"Rank badge, tapestry weave silk and metal-wrap...",This Ming Dynasty (1368-1644) rank badge is wo...,Rank badge for a sixth-rank civil official sho...,Purchased. Registered File number 1985/1393.,,https://framemark.vam.ac.uk/collections/2006AT...,https://framemark.vam.ac.uk/collections/2006AT...,V&A Collections (건별 라이선스 확인 필요)
1,YVAH02,흉배,빅토리아 앤 알버트 박물관,T.261-1929,,,Rank badge,https://collections.vam.ac.uk/item/O457659/,rank badge,Rank badge,...,tapestry,Length: 31.5cm; Width: 31cm,"Rank badge, silk tapestry (<i>kesi</i>), Qing ...",This is a rank badge for a court official of t...,"<i>Kesi</i> tapestry woven rank badge, with pe...",Registered File number 9901/1926.,Given by Mrs Lewis F. Day,https://framemark.vam.ac.uk/collections/2006BF...,https://framemark.vam.ac.uk/collections/2006BF...,V&A Collections (건별 라이선스 확인 필요)
2,YVAH03,흉배,빅토리아 앤 알버트 박물관,CIRC.37-1914,,,Rank badge,https://collections.vam.ac.uk/item/O457671/,rank badge,Rank badge,...,embroidering,Length: 28.5cm; Width: 30cm,"Rank badge, navy blue ground silk with silver ...",This is a rank badge for a court official of t...,Rank badge with silver and gilt thread embroid...,,Given by Miss Baxter in memory of Miss Kate Ba...,https://framemark.vam.ac.uk/collections/2023NJ...,https://framemark.vam.ac.uk/collections/2023NJ...,V&A Collections (건별 라이선스 확인 필요)
3,YVAH04,흉배,빅토리아 앤 알버트 박물관,CIRC.38-1914,,,Rank badge,https://collections.vam.ac.uk/item/O457670/,rank badge,Rank badge,...,embroidering,Length: 28.5cm; Width: 30cm,"Rank badge, navy blue ground silk with silver ...",This is a rank badge for a court official of t...,Rank badge with silver and gilt thread embroid...,,Given by Miss Baxter in memory of Miss Kate Ba...,NaN,,V&A Collections (건별 라이선스 확인 필요)
4,YVAH05,흉배,빅토리아 앤 알버트 박물관,T.243-1912,,,Rank badge,https://collections.vam.ac.uk/item/O457662/,rank badge,Rank badge,...,tapestry weave,Length: 29.5cm; Width: 29.5cm,"Rank badge, tapestry-weave silk (<i>kesi</i>),...",This is a rank badge for a court official of t...,Rank badge embroidered in seed stitches on a k...,"Given by Miss Baxter, accessioned in 1912. Thi...",Given by Miss Baxter,https://framemark.vam.ac.uk/collections/2012FG...,https://framemark.vam.ac.uk/collections/2012FG...,V&A Collections (건별 라이선스 확인 필요)
